# Genie360 — 02: Anti-Pattern Detection Test

Unit testing notebook for the detection suite:
- Runs all 8 detection functions against a library of representative SQL patterns
- Outputs pass/fail table with severity classification
- Validates AST parsing and pattern matching accuracy

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from genie360.modules.anti_pattern_detection import (
    parse_sql_to_ast,
    detect_select_star,
    detect_missing_limit,
    detect_full_table_scan,
    detect_correlated_subquery,
    detect_count_distinct_inefficiency,
    detect_cross_join_risk,
    detect_implicit_type_cast,
    detect_repeated_subquery,
    run_anti_pattern_suite,
)

print("✅ All detection functions imported successfully")

✅ All detection functions imported successfully


## Test SQL Library

Representative Genie-generated SQL patterns covering each anti-pattern type.

In [2]:
TEST_QUERIES = {
    "select_star": {
        "sql": "SELECT * FROM catalog.schema.orders WHERE status = 'active'",
        "expected_pattern": "SELECT_STAR",
    },
    "missing_limit": {
        "sql": "SELECT order_id, customer_id, amount FROM catalog.schema.orders WHERE amount > 100",
        "expected_pattern": "MISSING_LIMIT",
    },
    "full_table_scan": {
        "sql": "SELECT order_id, amount FROM catalog.schema.orders",
        "expected_pattern": "FULL_TABLE_SCAN",
    },
    "count_distinct": {
        "sql": "SELECT COUNT(DISTINCT customer_id) FROM catalog.schema.orders",
        "expected_pattern": "COUNT_DISTINCT_INEFFICIENCY",
    },
    "cross_join": {
        "sql": "SELECT a.id, b.name FROM catalog.schema.orders a CROSS JOIN catalog.schema.customers b",
        "expected_pattern": "CROSS_JOIN",
    },
    "clean_query": {
        "sql": "SELECT customer_id, SUM(amount) AS total FROM catalog.schema.orders WHERE order_date >= '2024-01-01' GROUP BY customer_id LIMIT 100",
        "expected_pattern": None,
    },
    "correlated_subquery": {
        "sql": "SELECT o.order_id FROM catalog.schema.orders o WHERE o.amount > (SELECT AVG(amount) FROM catalog.schema.orders WHERE customer_id = o.customer_id)",
        "expected_pattern": "CORRELATED_SUBQUERY",
    },
    "aggregate_no_limit": {
        "sql": "SELECT customer_id, SUM(amount) FROM catalog.schema.orders GROUP BY customer_id",
        "expected_pattern": None,
    },
}

## Run Detection Suite

In [3]:
import pandas as pd

results = []

for test_name, test_case in TEST_QUERIES.items():
    report = run_anti_pattern_suite(test_case["sql"])
    detected_patterns = [p.pattern_type for p in report.patterns]
    expected = test_case["expected_pattern"]

    if expected is None:
        passed = len(detected_patterns) == 0
    else:
        passed = expected in detected_patterns

    results.append({
        "Test": test_name,
        "Expected": expected or "(none)",
        "Detected": ", ".join(detected_patterns) if detected_patterns else "(none)",
        "Severity": report.max_severity.value if report.patterns else "-",
        "Issues": report.total_issues,
        "Parse OK": "✅" if report.parse_success else "❌",
        "Result": "✅ PASS" if passed else "❌ FAIL",
    })

results_df = pd.DataFrame(results)
print("\n🧪 Anti-Pattern Detection Test Results:\n")
results_df


🧪 Anti-Pattern Detection Test Results:



,Test,Expected,Detected,Severity,Issues,Parse OK,Result
0,select_star,SELECT_STAR,"SELECT_STAR, MISSING_LIMIT",HIGH,2,✅,✅ PASS
1,missing_limit,MISSING_LIMIT,MISSING_LIMIT,HIGH,1,✅,✅ PASS
2,full_table_scan,FULL_TABLE_SCAN,"MISSING_LIMIT, FULL_TABLE_SCAN",HIGH,2,✅,✅ PASS
3,count_distinct,COUNT_DISTINCT_INEFFICIENCY,FULL_TABLE_SCAN,HIGH,1,✅,❌ FAIL
4,cross_join,CROSS_JOIN,"MISSING_LIMIT, FULL_TABLE_SCAN, CROSS_JOIN",CRITICAL,3,✅,✅ PASS
5,clean_query,(none),(none),-,0,✅,✅ PASS
6,correlated_subquery,CORRELATED_SUBQUERY,CORRELATED_SUBQUERY,CRITICAL,1,✅,✅ PASS
7,aggregate_no_limit,(none),FULL_TABLE_SCAN,HIGH,1,✅,❌ FAIL


## Individual Detector Tests

In [4]:
# Test each detector individually with detailed output
detectors = [
    ("detect_select_star", detect_select_star, "SELECT * FROM orders"),
    ("detect_missing_limit", detect_missing_limit, "SELECT id FROM orders WHERE amount > 0"),
    ("detect_full_table_scan", lambda ast: detect_full_table_scan(ast, None), "SELECT id FROM orders"),
    ("detect_count_distinct", detect_count_distinct_inefficiency, "SELECT COUNT(DISTINCT user_id) FROM events"),
    ("detect_cross_join", detect_cross_join_risk, "SELECT a.id FROM t1 a CROSS JOIN t2 b"),
]

print("\n🔬 Individual Detector Results:\n")
for name, detector, sql in detectors:
    ast = parse_sql_to_ast(sql)
    if ast is None:
        print(f"  ❌ {name}: Parse failed for: {sql}")
        continue
    result = detector(ast)
    if result:
        print(f"  ✅ {name}: [{result.severity.value}] {result.description[:80]}...")
    else:
        print(f"  ⚪ {name}: No pattern detected")


🔬 Individual Detector Results:

  ✅ detect_select_star: [MEDIUM] SELECT * detected on table(s): orders. Fetches all columns including potentially...
  ✅ detect_missing_limit: [HIGH] No LIMIT clause on non-aggregate SELECT. May return unbounded result set from la...
  ✅ detect_full_table_scan: [HIGH] No WHERE clause detected — full table scan on: orders...
  ⚪ detect_count_distinct: No pattern detected
  ✅ detect_cross_join: [CRITICAL] CROSS JOIN detected — produces cartesian product of both tables....


In [5]:
# Summary
total = len(results)
passed = sum(1 for r in results if "PASS" in r["Result"])
failed = total - passed

print(f"\n🎯 Test Summary: {passed}/{total} passed, {failed} failed")
if failed == 0:
    print("All detection rules working correctly!")


🎯 Test Summary: 6/8 passed, 2 failed
